In [27]:
import osmnx
import pandas as pd
import janitor
import numpy as np

pd.set_option('display.max_columns', 110)

In [15]:
# https://wiki.openstreetmap.org/wiki/Tag:shop%3Dmall
payload = osmnx.features_from_place("Singapore", tags={'shop': 'mall'})
payload

geometry  \
element id                                                              
node    410463765                             POINT (103.78838 1.308)   
        4233370769                          POINT (103.84965 1.29818)   
        4233370770                          POINT (103.85076 1.29922)   
        4682788852                          POINT (103.90323 1.30313)   
        6440064328                          POINT (103.86857 1.33144)   
...                                                               ...   
way     1242216910  POLYGON ((103.8144 1.32264, 103.81442 1.32276,...   
        1242216911  POLYGON ((103.81444 1.32263, 103.81446 1.32276...   
        1294482731  POLYGON ((103.83916 1.29217, 103.83948 1.2926,...   
        1385980996  POLYGON ((103.846 1.28358, 103.84607 1.2837, 1...   
        1395633020  POLYGON ((103.78546 1.4356, 103.78543 1.43563,...   

                    addr:city addr:country  addr:housename addr:housenumber  \
element id                                                                    
node    410463765   Singapore           SG  The Star Vista                1   
        4233370769        NaN          NaN             NaN              NaN   
        4233370770        NaN          NaN             NaN              NaN   
        4682788852        NaN          NaN             NaN              NaN   
        6440064328  Singapore          NaN             NaN               51   
...                       ...          ...             ...              ...   
way     1242216910  Singapore          NaN             NaN              501   
        1242216911  Singapore          NaN             NaN              501   
        1294482731  Singapore           SG             NaN               46   
        1385980996        NaN          NaN             NaN              NaN   
        1395633020        NaN          NaN             NaN                1   

                   addr:postcode           addr:street bench  bin building  \
element id                                                                   
node    410463765         138617  Vista Exchange Green   yes  yes   retail   
        4233370769           NaN                   NaN   NaN  NaN      NaN   
        4233370770           NaN                   NaN   NaN  NaN      NaN   
        4682788852           NaN    Marine Parade Road   NaN  NaN      NaN   
        6440064328        347697  Upper Serangoon Road   NaN  NaN      NaN   
...                          ...                   ...   ...  ...      ...   
way     1242216910        259760      Bukit Timah Road   NaN  NaN   retail   
        1242216911        259760      Bukit Timah Road   NaN  NaN   retail   
        1294482731        239351          Kim Yam Road   NaN  NaN   retail   
        1385980996           NaN                   NaN   NaN  NaN      NaN   
        1395633020        738099      Woodlands Square   NaN  NaN   retail   

                   building:levels building:part  bus height   highway  \
element id                                                               
node    410463765               11            no  yes      0  bus_stop   
        4233370769             NaN           NaN  NaN    NaN       NaN   
        4233370770             NaN           NaN  NaN    NaN       NaN   
        4682788852             NaN           NaN  NaN    NaN       NaN   
        6440064328             NaN           NaN  NaN    NaN       NaN   
...                            ...           ...  ...    ...       ...   
way     1242216910               2           NaN  NaN    NaN       NaN   
        1242216911               2           NaN  NaN    NaN       NaN   
        1294482731             NaN           NaN  NaN    NaN       NaN   
        1385980996             NaN           NaN  NaN    NaN       NaN   
        1395633020               7           NaN  NaN    NaN       NaN   

                                                                image  lit  \
element id                 

In [33]:
priority_order = {
   'way': 1,      # Building footprints - most accurate for mall boundaries
   'node': 2,     # Point locations - simple but less detailed
   'relation': 3  # Complex structures - often contain duplicate data from member ways
}

gdf_malls = (
    payload
    .clean_names()
    .reset_index()
    .query("addr_housename==addr_housename | name==name")
    .assign(name=lambda df_: np.where(df_["name"].isna(), df_["addr_housename"], df_["name"]))
    # Dedup
    .assign(priority=lambda df_: df_['element'].map(priority_order))
    .sort_values('priority')
    .drop_duplicates(subset=['name'], keep='first', ignore_index=True)
    .remove_columns('priority')
)
gdf_malls

/home/lsys/nbhmv/venv_nbhmv/lib/python3.10/site-packages/pandas_flavor/register.py:164: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.drop` instead.
  return method(self._obj, *args, **kwargs)


,element,id,geometry,addr_city,addr_country,addr_housename,addr_housenumber,addr_postcode,addr_street,bench,bin,building,building_levels,building_part,bus,height,highway,image,lit,location,mapillary,name,name_en,name_zh,operator,public_transport,ref,route_ref,shelter,shop,tactile_paving,wheelchair,wikidata,air_conditioning,check_date,level,currency_sgd,fixme,opening_hours,payment_cash,payment_credit_cards,payment_debit_cards,website,source,wikimedia_commons,wikipedia,building_colour,building_material,email,internet_access,layer,name_ko,name_zh_hans,name_zh_hant,phone,roof_colour,roof_material,start_date,stroller,building_min_level,building_levels_underground,toilets,access,fee,old_name,parking,foot,landuse,internet_access_fee,name_ar,construction,note,addr_neighbourhood,name_ms,office,toilets_wheelchair,operator_wikidata,designation,amenity,alt_name,roof_shape,underground,toilets_level,toilets_panoramax,roof_levels,min_height,residential,description,baby_feeding,contact_facebook,contact_instagram,payment_contactless,full_name,short_name,fax,covered,survey_date,train_station_name,aeroway,short_name_zh,building_use,postal_code,opening_date,roof_height,old_name_1983_1995,old_name_1995_2011,type,architect
0,way,32774317,"POLYGON ((103.90116 1.38506, 103.90151 1.38552...",NaN,NaN,NaN,118,540118,Rivervale Drive,NaN,NaN,yes,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rivervale Plaza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mall,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,way,26561741,"POLYGON ((103.99431 1.36489, 103.99427 1.36494...",Singapore,SG,NaN,80,819642,Airport Boulevard,NaN,NaN,yes,3,NaN,NaN,30,NaN,NaN,NaN,NaN,NaN,Changi Airport Terminal 1,Changi Airport Terminal 1,樟宜机场1号航站楼,NaN,NaN,NaN,NaN,NaN,mall,NaN,yes,Q55721632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1981,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Terminal 1,NaN,NaN,NaN,NaN,terminal,1号航站楼,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,way,32416309,"POLYGON ((103.8457 1.37205, 103.84619 1.3721, ...",Singapore,SG,Broadway Plaza,4190,569841,Ang Mo Kio Avenue 6,NaN,NaN,retail,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Broadway Plaza,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mall,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,way,30527021,"POLYGON ((103.85308 1.30254, 103.85302 1.30259...",Singapore,SG,NaN,1,188504,Rochor Canal Road,NaN,NaN,retail,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sim Lim Square,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mall,NaN,limited,Q3483927,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Category:Sim Lim Square,en:Sim Lim Square,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,way,32774374,"POLYGON ((103.90426 1.39265, 103.90399 1.39227...",Singapore,NaN,NaN,11,545082,Rivervale Crescent,NaN,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rivervale Mall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mall,NaN,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [36]:
community_tags = {
    'amenity': ['community_centre', 'social_facility'],
#     'leisure': ['sports_centre', 'fitness_centre'],
#     'club': True,  # Generic club tag
    'building': ['community_centre', 'club']
}

# Query it
community_centres = osmnx.features_from_place("Singapore", tags=community_tags)
community_centres

geometry  \
element id                                                              
node    1565963296                          POINT (103.86187 1.30974)   
        2491836852                          POINT (103.76872 1.30969)   
        2870022427                            POINT (103.79065 1.343)   
        2990751442                          POINT (103.88967 1.35488)   
        2990751443                          POINT (103.84554 1.35865)   
...                                                               ...   
way     1342044000  POLYGON ((103.89673 1.30465, 103.89683 1.30431...   
        1346368010  POLYGON ((103.91299 1.32125, 103.91293 1.32156...   
        1346368010  POLYGON ((103.91299 1.32125, 103.91293 1.32156...   
        1367629249  POLYGON ((103.89446 1.403, 103.89444 1.40287, ...   
        1392539057  POLYGON ((103.84669 1.42441, 103.84669 1.42431...   

                    addr:city addr:housenumber addr:postcode  \
element id                                                     
node    1565963296  Singapore             318C        208563   
        2491836852        NaN              NaN           NaN   
        2870022427        NaN              NaN           NaN   
        2990751442  Singapore              110        530110   
        2990751443  Singapore              233        570233   
...                       ...              ...           ...   
way     1342044000        NaN              NaN           NaN   
        1346368010        NaN              NaN           NaN   
        1346368010        NaN              NaN           NaN   
        1367629249  Singapore                1        829879   
        1392539057        NaN              NaN           NaN   

                             addr:street           amenity  \
element id                                                   
node    1565963296  King George's Avenue  community_centre   
        2491836852                   NaN  community_centre   
        2870022427                   NaN   social_facility   
        2990751442      Hougang Avenue 1  community_centre   
        2990751443      Bishan Street 22  community_centre   
...                                  ...               ...   
way     1342044000                   NaN   social_facility   
        1346368010                   NaN  community_centre   
        1346368010                   NaN  community_centre   
        1367629249           Sumang Walk   social_facility   
        1392539057                   NaN  community_centre   

                                                       name  \
element id                                                    
node    1565963296                      Makespace Singapore   
        2491836852       Casa Clementi Residents' Committee   
        2870022427                               Swiss Club   
        2990751442                Aljunied Community Centre   
        2990751443            Bishan North Community Centre   
...                                                     ...   
way     1342044000        THK Nursing Home @ Tanjong Katong   
        1346368010         Kampong Kembangan Community Club   
        1346368010         Kampong Kembangan Community Club   
        1367629249  St Luke’s Eldercare Residence @ Punggol   
        1392539057                             N4 Courtyard   

                                  website addr:country  \
element id                                               
node    1565963296  https://makespace.sg/          NaN   
        2491836852                    NaN          NaN   
        2870022427                    NaN          NaN   
        2990751442                    NaN           SG   
        2990751443                    NaN           SG   
...                                   ...          ...   
way     1342044000                    NaN          NaN   
        1346368010                    NaN          NaN   
        1346368010                    NaN          NaN   
        13676